In [ ]:
import pandas as pd

data = {
    "text": [
        "Verify your account immediately at http://fakebank.com",
        "Meeting scheduled tomorrow at 10 AM",
        "Click here to update your password",
        "Project report attached",
        "Urgent! Login to secure your bank account",
        "Thank you for attending the meeting"
    ],
    "label": [1, 0, 1, 0, 1, 0]
}

df = pd.DataFrame(data)
df.to_csv("emails.csv", index=False)

print("Dataset created successfully!")



import re
from urllib.parse import urlparse

from sklearn.model_selection import train_test_split
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


df = pd.read_csv("emails.csv")


class URLFeatures(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        features = []

        phishing_keywords = [
            "verify", "login", "account",
            "password", "bank", "urgent",
            "click", "update"
        ]

        for text in X:
            urls = re.findall(r'https?://\S+|www\.\S+', str(text))

            num_urls = len(urls)

            suspicious_url = 0

            for url in urls:
                try:
                    domain = urlparse(url).netloc
                    if "-" in domain or len(domain) > 25:
                        suspicious_url = 1
                except:
                    pass

            keyword_count = sum(
                text.lower().count(word)
                for word in phishing_keywords
            )

            features.append([
                num_urls,
                suspicious_url,
                keyword_count
            ])

        return features


combined_features = FeatureUnion([

    ("text_features",
     TfidfVectorizer(
         stop_words='english',
         max_features=5000
     )),

    ("url_features",
     URLFeatures())

])


model = Pipeline([
    ("features", combined_features),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ))
])


X_train, X_test, y_train, y_test = train_test_split(
    df["text"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)


model.fit(X_train, y_train)


y_pred = model.predict(X_test)


accuracy = accuracy_score(y_test, y_pred)

print(f"\nAccuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


sample_emails = [

    "URGENT! Verify your bank account immediately at http://secure-login-update.com",

    "Team meeting moved to Friday. Please confirm attendance.",

    "Your mailbox storage is full. Click here to update your account.",

    "Monthly sales report attached."
]

predictions = model.predict(sample_emails)

for email, pred in zip(sample_emails, predictions):
    label = "Phishing" if pred == 1 else "Safe"
    print(f"\nEmail: {email}")
    print("Prediction:", label)

Dataset created successfully!

Accuracy: 1.0000

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         1

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2


Confusion Matrix:
[[1 0]
 [0 1]]

Email: URGENT! Verify your bank account immediately at http://secure-login-update.com
Prediction: Phishing

Email: Team meeting moved to Friday. Please confirm attendance.
Prediction: Safe

Email: Your mailbox storage is full. Click here to update your account.
Prediction: Phishing

Email: Monthly sales report attached.
Prediction: Safe
